#### 문장의 유사도 분석
: 두 개의 문장이 비슷한 것인지 또는, 관련이 있는 것인지를 분석하는 것

> LLM 모델은 프롬프트에 입력하는 언어에 따른 성능 차이가 없다 -> 어차피 문장을 전부 숫자 데이터로 변환하기 때문

#### 코사인(Cosine) 유사도

In [30]:
a = '딥러닝은 매우 재미있는 기술이라 공부하고 있다'
# b = '공부하면 재미있는 기술이라 배우고 있다'
# b = '01.DeepLearning 개요.ipynb'
b = '딥러닝은 매우 재미있는 기술이라 공부했냐'

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [31]:
sentences = (a, b)
tfid_vactorizer = TfidfVectorizer()

In [32]:
sentences

('딥러닝은 매우 재미있는 기술이라 공부하고 있다', '딥러닝은 매우 재미있는 기술이라 공부했냐')

In [33]:
# 문장 벡터화 하기 : 사전 만들기
tfid_matrix = tfid_vactorizer.fit_transform(sentences)

In [34]:
from sklearn.metrics.pairwise import cosine_similarity
# 첫번째 문장과 두번째 문장 비교

cos_similarity = cosine_similarity(tfid_matrix[0:1], tfid_matrix[1:2]) # 2차원 값으로 받기 때문에 범위로 지정해 줌
print("코사인 유사도 측정:", cos_similarity)

코사인 유사도 측정: [[0.58033298]]


---
#### 유사도를 이용한 추천 시스템 구현
: 코사인 유사도만으로 영화의 줄거리에 기반하여 영화를 추천하는 시스템

In [35]:
import pandas as pd

In [39]:
data = pd.read_csv('../Data/movies_metadata.csv', low_memory=False)
data[['title', 'overview']].head(2)

,title,overview
0,Toy Story,"Led by Woody, Andy's toys live happily in his ..."
1,Jumanji,When siblings Judy and Peter discover an encha...


In [40]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

In [41]:
# 상위 2만개만 사용
data = data.head(20000)

In [42]:
data['overview'].isnull().sum()

np.int64(135)

In [43]:
# 결측치를 빈값으로 대체
data['overview'] = data['overview'].fillna("")

In [45]:
# 행렬 크기 구하기
tfidf = TfidfVectorizer(stop_words='english')
tfid_matrix = tfidf.fit_transform(data['overview'])
tfid_matrix.shape

(20000, 47487)

In [46]:
# 학습
cosine_sim = cosine_similarity(tfid_matrix, tfid_matrix)
cosine_sim.shape

(20000, 20000)

In [47]:
cosine_sim

array([[1.        , 0.01575748, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.01575748, 1.        , 0.04907345, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.04907345, 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.        ,
        0.08375766],
       [0.        , 0.        , 0.        , ..., 0.        , 1.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.08375766, 0.        ,
        1.        ]], shape=(20000, 20000))

In [48]:
# 기존 데이터 프레임으로 부터 영화 타이틀을 key, 영화 index 를 value 로 하는 딕셔너리
title_to_index = dict(zip(data['title'], data.index))

In [49]:
title_to_index['Toy Story']

0

In [50]:
# 선택한 영화의 제목을 입력하면 Cos 유사도를 통해 가장 Overview가 유사한 10개의 영화를 찾아내는 함수

def get_recommendation(title, cosine_sim=cosine_sim):
    # 선택한 영화의 타이틀로부터 해당 영화의 인덱스를 받아온다.
    idx = title_to_index[title]
    # 해당 영화와 모든 영화와의 유사도를 가져온다.
    sim_scores = list(enumerate(cosine_sim[idx]))
    # 유사도에 따라 영화들을 정렬한다.
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    # 가장 유사한 10개의 영화 가져오기
    sim_scores = sim_scores[1:11]
    # 가장 유사한 10개의 영화 인덱스를 가져오기
    movies_indices = [idx[0] for idx in sim_scores]
    # 가장 유사한 10개의 영화 제목 Return
    return data['title'].iloc[movies_indices]

In [52]:
get_recommendation("Toy Story")

15348               Toy Story 3
2997                Toy Story 2
10301    The 40 Year Old Virgin
8327                  The Champ
1071      Rebel Without a Cause
11399    For Your Consideration
1932                  Condorman
3057            Man on the Moon
485                      Malice
11606              Factory Girl
Name: title, dtype: object